In [1]:
import pandas as pd

df = pd.read_csv("https://github.com/ellimilial/clinical-trials-negative-efficacy/raw/refs/heads/main/01-all-clinical-trials.csv.zip")
print(f"Loaded a total of {len(set(df.nct_id))} trials")
print(f"  - {len(df)} trial-intervention-condition")
print(f"  - {len(set(df.condition_name))} conditions")
print(f"  - {len(set(df.intervention_name))} interventions")

df.describe(include='all')

Loaded a total of 582141 trials
  - 1944960 trial-intervention-condition
  - 129379 conditions
  - 513773 interventions


,nct_id,brief_title,official_title,overall_status,start_date,completion_date,study_type,phase,why_stopped,condition_name,intervention_name,intervention_type
count,1944960,1944960,1920708,1944960,1933161,1874901,1943989,900713,177518,1943950,1835012,1835284
unique,582141,579644,568025,14,9212,10310,3,7,32998,129378,513772,11
top,NCT03878524,Serial Measurements of Molecular and Architect...,Serial Measurements of Molecular and Architect...,COMPLETED,2014-01-31,2026-12-31,INTERVENTIONAL,PHASE2,Low accrual,Healthy,Placebo,DRUG
freq,2800,2800,2800,990709,6317,32590,1623356,298082,3869,21875,53986,686124


In [2]:
# Only evaluate stopping reasons where we can link to both condition and intervention.
df_filtered = df[pd.notna(df["why_stopped"]) & pd.notna(df["condition_name"]) & pd.notna(df["intervention_name"])]
# Since the model was trained on case-sensitive data we should not be collapsing by case.
df_filtered["why_stopped"].drop_duplicates().reset_index(drop=True)

0                               Replaced by another study.
1          Study never opened; never enrolled participants
2                       Original P.I. left the institution
3            Unable to recruit adequate number of subjects
4                                  Study was never funded.
                               ...                        
31114     etik onay alındığı tarihte yapıldı ve tamamlandı
31115    Participant recruitment has been stopped becau...
31116    This study has been transferred to the Human R...
31117    Recruitment was temporarily suspended pending ...
31118    Operational decision not to proceed at this st...
Name: why_stopped, Length: 31119, dtype: object

In [3]:
import json
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import expit, softmax
from tqdm.auto import tqdm

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
MODEL_ID = "ellimilial/clinical-trials-negative-efficacy-biomedbert"
# MODEL_ID = "ellimilial/clinical-trials-negative-efficacy-deberta-v3-base"
TEXT_COL = "why_stopped"
BATCH_SIZE = 32

device = (
    "mps"
    if torch.backends.mps.is_built() and torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# -------------------------------------------------------------------
# Load model + published inference settings
# -------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

config_path = hf_hub_download(MODEL_ID, "inference_config.json")
with open(config_path) as f:
    inference_config = json.load(f)

id2label = {int(k): v for k, v in inference_config["id2label"].items()}
positive_label_id = int(inference_config["positive_label"])
negative_label_id = int(inference_config["negative_label"])
threshold = float(inference_config["threshold"])
max_length = int(inference_config["max_length"])

print("Model labels:", id2label)
print("Inference config:", {
    "threshold": threshold,
    "positive_label": positive_label_id,
    "negative_label": negative_label_id,
    "max_length": max_length,
})

# -------------------------------------------------------------------
# Prepare unique stopping reasons
# -------------------------------------------------------------------
why_stopped_unique = (
    df_filtered[[TEXT_COL]]
    .drop_duplicates()
    .reset_index(drop=True)
)

texts = why_stopped_unique[TEXT_COL].astype(str).tolist()

# -------------------------------------------------------------------
# Predict
# -------------------------------------------------------------------
all_probs = []
all_preds = []
all_argmax_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch_texts = texts[i : i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        outputs = model(**inputs)
        logits = outputs.logits.detach().cpu().numpy()

        # Use the published positive class and operating threshold.
        if logits.shape[1] == 1:
            probs_positive = expit(logits[:, 0])
            argmax_preds = np.where(
                probs_positive >= 0.5, positive_label_id, negative_label_id
            )
        else:
            probs = softmax(logits, axis=1)
            probs_positive = probs[:, positive_label_id]
            argmax_preds = probs.argmax(axis=1)

        preds = np.where(
            probs_positive >= threshold, positive_label_id, negative_label_id
        )

        all_probs.extend(probs_positive.tolist())
        all_preds.extend(preds.tolist())
        all_argmax_preds.extend(argmax_preds.tolist())

why_stopped_predictions = why_stopped_unique.copy()
why_stopped_predictions["y_prob"] = all_probs
why_stopped_predictions["y_pred"] = all_preds
why_stopped_predictions["argmax_pred"] = all_argmax_preds
why_stopped_predictions["decision_threshold"] = threshold
why_stopped_predictions["predicted_label"] = [
    id2label.get(int(pred), str(pred)) for pred in all_preds
]
why_stopped_predictions["argmax_label"] = [
    id2label.get(int(pred), str(pred)) for pred in all_argmax_preds
]

why_stopped_predictions.head()

Using device: mps


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model labels: {0: 'not_negative_efficacy', 1: 'negative_efficacy'}
Inference config: {'threshold': 0.10404040404040403, 'positive_label': 1, 'negative_label': 0, 'max_length': 96}


  0%|          | 0/973 [00:00<?, ?it/s]

,why_stopped,y_prob,y_pred,argmax_pred,decision_threshold,predicted_label,argmax_label
0,Replaced by another study.,0.000096,0,0,0.10404,not_negative_efficacy,not_negative_efficacy
1,Study never opened; never enrolled participants,0.000092,0,0,0.10404,not_negative_efficacy,not_negative_efficacy
2,Original P.I. left the institution,0.000099,0,0,0.10404,not_negative_efficacy,not_negative_efficacy
3,Unable to recruit adequate number of subjects,0.000091,0,0,0.10404,not_negative_efficacy,not_negative_efficacy
4,Study was never funded.,0.000102,0,0,0.10404,not_negative_efficacy,not_negative_efficacy


In [4]:

df_scored = df_filtered.merge(
    why_stopped_predictions,
    on="why_stopped",
    how="left",
)

df_scored[
    [
        "nct_id",
        "condition_name",
        "intervention_name",
        "why_stopped",
        "y_prob",
        "y_pred",
        "predicted_label",
    ]
].head()

,nct_id,condition_name,intervention_name,why_stopped,y_prob,y_pred,predicted_label
0,NCT00000105,Cancer,Biosyn KLH,Replaced by another study.,0.000096,0,not_negative_efficacy
1,NCT00000105,Cancer,Intracel KLH Vaccine,Replaced by another study.,0.000096,0,not_negative_efficacy
2,NCT00000105,Cancer,Montanide ISA51,Replaced by another study.,0.000096,0,not_negative_efficacy
3,NCT00000105,Cancer,Tetanus toxoid,Replaced by another study.,0.000096,0,not_negative_efficacy
4,NCT00000270,Cocaine-Related Disorders,Cocaine,Study never opened; never enrolled participants,0.000092,0,not_negative_efficacy


In [5]:
why_stopped_predictions.to_csv("unique_why_stopped_predictions.csv", index=False)
df_scored.to_csv("02-negative-efficacy-prediction.csv.zip", index=False, compression="zip")

print("Saved:")
print(" - unique_why_stopped_predictions.csv")
print(" - 02-negative-efficacy-prediction.csv.zip")

Saved:
 - unique_why_stopped_predictions.csv
 - 02-negative-efficacy-prediction.csv.zip
